# Coordination analysis

Reads `data/derived` tables, joins them to comuna boundaries, prints summaries.

Maps render interactively from a **GeoDataFrame** via `.explore()` — no PNG files saved.

**Setup** (once, in your notebook kernel venv):

```bash
pip install -r requirements-notebook.txt
```

Or: `pip install geopandas pandas "folium>=0.12" branca matplotlib mapclassify`

In [26]:
import csv
from collections import Counter, defaultdict
from pathlib import Path

import geopandas as gpd
import pandas as pd

# Quick check — re-run `pip install -r requirements-notebook.txt` if this fails
try:
    import branca  # noqa: F401
    import folium  # noqa: F401
    import mapclassify  # noqa: F401
except ImportError as e:
    raise ImportError(
        "Missing map deps. Run: pip install -r requirements-notebook.txt"
    ) from e

BASE = Path("..").resolve()
DERIVED = BASE / "data" / "derived"
GEOJSON = BASE / "data" / "input" / "raw_data_cl_ocha_ab.geojson"

## 1. Load derived tables

In [27]:
def read_csv(filename):
    with open(DERIVED / filename, encoding="utf-8") as f:
        return list(csv.DictReader(f))

units = read_csv("coordination_units.csv")
bundles = read_csv("unit_bundle_candidates.csv")
funders = read_csv("valdivia_funders_open.csv")
matches = read_csv("valdivia_action_matches.csv")

print(len(units), "unit rows")
print(len(bundles), "bundle candidates")
print(len(funders), "funders for Valdivia")
print(len(matches), "action matches for Valdivia")

345 unit rows
634 bundle candidates
78 funders for Valdivia
102 action matches for Valdivia


## 2. Comuna GeoDataFrame (geometry + unit attributes)

In [28]:
unit_cols = [
    "locode", "comuna", "region", "unit_id", "unit_anchor",
    "unit_viable_anchor", "is_anchor", "anchor_score", "cofinance_score",
]
unit_df = pd.DataFrame(units)[unit_cols].drop_duplicates("locode")

gdf = gpd.read_file(GEOJSON)
gdf = gdf.merge(unit_df, on="locode", how="left")

gdf["pool_status"] = gdf["unit_viable_anchor"].map(
    {"True": "viable anchor", "False": "needs TA"}
).fillna("not scored")

print(gdf[["comuna_name", "locode", "unit_id", "unit_anchor", "pool_status"]].head())
print(f"\n{len(gdf)} comunas on map · {gdf['locode'].notna().sum()} with unit assignment")

     comuna_name  locode unit_id unit_anchor    pool_status
0        Iquique  CL IQQ      18     Iquique  viable anchor
1  Alto Hospicio  CL AHP      18     Iquique  viable anchor
2      Tocopilla     NaN     NaN         NaN     not scored
3         Camiña  CL CMA      18     Iquique  viable anchor
4       Colchane  CL CNE      18     Iquique  viable anchor

345 comunas on map · 314 with unit assignment


## 3. Summary

In [29]:
by_unit = defaultdict(list)
for row in units:
    if row["region"] and row["region"] != "region":
        by_unit[row["unit_id"]].append(row)

viable_units = [
    uid for uid, members in by_unit.items()
    if any(m["unit_viable_anchor"] == "True" for m in members)
]
feasible_bundles = [b for b in bundles if b["bundle_feasible"] == "True"]

print("National coordination")
print(f"  units: {len(by_unit)} ({len(viable_units)} with viable anchor)")
print(f"  feasible bundles: {len(feasible_bundles)}")

print("\nValdivia")
for role in ("applicant", "facilitator", "referrer"):
    n = sum(1 for r in funders if r["role"] == role)
    print(f"  {role}: {n} funders")

n_match = sum(1 for r in matches if r["verdict"] == "match")
print(f"  action matches: {n_match} direct, {len(matches) - n_match} referrer-route")

National coordination
  units: 77 (68 with viable anchor)
  feasible bundles: 308

Valdivia
  applicant: 52 funders
  facilitator: 1 funders
  referrer: 25 funders
  action matches: 75 direct, 27 referrer-route


## 4. Coordination units (text)

In [30]:
print(f"{'unit':>4}  {'anchor':<22}  {'size':>4}  {'viable':>6}  members")
print("-" * 90)

for uid in sorted(by_unit, key=lambda x: int(x)):
    members = by_unit[uid]
    anchor = members[0]["unit_anchor"]
    viable = members[0]["unit_viable_anchor"]
    names = ", ".join(m["comuna"] for m in members)
    print(f"{uid:>4}  {anchor:<22}  {len(members):>4}  {viable:>6}  {names}")

no_anchor = [uid for uid, m in by_unit.items() if m[0]["unit_viable_anchor"] != "True"]
print(f"\n{len(by_unit)} units · {len(no_anchor)} without viable anchor: {', '.join(no_anchor)}")

unit  anchor                  size  viable  members
------------------------------------------------------------------------------------------
   0  Providencia                3    True  Providencia, Ñuñoa, Santiago
   1  Lo Barnechea               6    True  Lo Barnechea, Vitacura, Huechuraba, Colina, San José de Maipo, Las Condes
   2  Valparaíso                 5    True  Valparaíso, Quilpué, Viña del Mar, Casablanca, Olmué
   3  Tomé                       3    True  Tomé, Penco, Florida
   4  San Antonio                3    True  San Antonio, Santo Domingo, Cartagena
   5  Lampa                      5    True  Lampa, Pudahuel, Curacaví, Tiltil, María Pinto
   6  Macul                      3    True  Macul, La Florida, San Joaquín
   7  Concepción                 7    True  Concepción, San Pedro de la Paz, Chiguayante, Hualpén, Hualqui, Talcahuano, San Rosendo
   8  La Serena                  6    True  La Serena, Coquimbo, Vicuña, Andacollo, La Higuera, Paihuano
   9  Coyhaique    

## 5. National map — pool viability

Interactive map from the GeoDataFrame (pan/zoom in the notebook).

In [31]:
gdf.explore(
    column="pool_status",
    categorical=True,
    legend=True,
    tooltip=["comuna_name", "unit_anchor", "pool_status", "anchor_score", "cofinance_score"],
    tiles="CartoDB positron",
)

## 6. Valdivia pool (unit 11) — Los Ríos

Valdivia anchor + five neighbouring comunas in the demo pool.

In [32]:
POOL = {"Valdivia", "Paillaco", "Los Lagos", "Corral", "Máfil", "Mariquina"}
LOS_RIOS = "Región de Los Ríos"

pool = gdf[gdf["region_name"] == LOS_RIOS].copy()
pool["pool_role"] = pool["comuna_name"].map(
    lambda n: "anchor" if n == "Valdivia" else ("pool member" if n in POOL else "other")
)

print(pool[["comuna_name", "pool_role", "cofinance_score", "unit_anchor"]].to_string(index=False))

pool.explore(
    column="pool_role",
    categorical=True,
    legend=True,
    tooltip=["comuna_name", "pool_role", "cofinance_score", "unit_anchor"],
    tiles="CartoDB positron",
)

comuna_name   pool_role cofinance_score unit_anchor
   Valdivia      anchor            65.7    Valdivia
     Corral pool member            10.7    Valdivia
      Lanco       other            21.5 Panguipulli
  Los Lagos pool member            42.8    Valdivia
      Máfil pool member            25.5    Valdivia
  Mariquina pool member             NaN         NaN
   Paillaco pool member            43.5    Valdivia
Panguipulli       other            49.4 Panguipulli
   La Unión       other            57.6  Lago Ranco
    Futrono       other            59.9 Panguipulli
 Lago Ranco       other            78.6  Lago Ranco
  Río Bueno       other             NaN         NaN


## 7. Feasible bundles by sector (text)

In [33]:
feasible = [b for b in bundles if b["bundle_feasible"] == "True"]
rows = []
for sector in sorted({b["sector"] for b in feasible}):
    for tier in ("highly_coordinated", "semi_coordinated", "idiosyncratic"):
        n = sum(1 for b in feasible if b["sector"] == sector and b["coordination_tier"] == tier)
        rows.append({"sector": sector, "tier": tier, "feasible_bundles": n})

bundle_table = pd.DataFrame(rows).pivot(index="sector", columns="tier", values="feasible_bundles").fillna(0).astype(int)
bundle_table["total"] = bundle_table.sum(axis=1)
print(bundle_table.to_string())
print(f"\ntotal feasible bundles: {int(bundle_table['total'].sum())}")

tier               highly_coordinated  idiosyncratic  semi_coordinated  total
sector                                                                       
Stationary Energy                  62              0                62    124
Transportation                     49              0                49     98
Waste                              43              0                43     86

total feasible bundles: 308


## 8. Insights — reading the methodology from the data

Six small, self-contained cells. Each one turns one idea from `research/` into something you
can see in the numbers. Run them top to bottom; every cell ends with a plain-language **Insight:**.

In [34]:
# These insight cells use two more derived tables, so load them once here.
action_tiers = read_csv("action_coordination.csv")    # each plan action -> a coordination tier
capacity     = read_csv("comuna_capacity_scores.csv")  # per-comuna capacity / co-finance

print(len(action_tiers), "actions tagged with a coordination tier")
print(len(capacity), "comunas with capacity scores")

102 actions tagged with a coordination tier
314 comunas with capacity scores


### Insight 1 — The bundle funnel: why 634 candidates become 308 deals

A candidate is only *feasible* if it clears **three gates** (`research/06`). Tagging each candidate
with the **first** gate it fails shows the methodology as a funnel — every drop is a real-world reason
the money can't move.

In [37]:
# Gate order matters so reasons don't double-count: tier -> actor -> anchor.
def drop_reason(b):
    if b["coordination_tier"] == "idiosyncratic":
        return "fails gate 1: not poolable (idiosyncratic action)"
    if b["actor_route"] != "municipal applicant":
        return "fails gate 2: firm-actor (city can only refer)"
    if b["viable_anchor"] != "True":
        return "fails gate 3: no viable anchor (needs TA into pool)"
    return "FEASIBLE (clears all three)"

funnel = Counter(drop_reason(b) for b in bundles)
for reason in sorted(funnel):
    print(f"{funnel[reason]:>4}  {reason}")

print(f"\nInsight: of {len(bundles)} candidates, {funnel['FEASIBLE (clears all three)']} survive all three gates.")
print("The gates ARE the methodology — each drop names why that bundle isn't bankable.")

NameError: name 'Counter' is not defined

### Insight 2 — The actor gap: the #1 error mode, inside Valdivia's own plan

A fund can match an action on *sector* yet be useless because the eligible applicant is a
firm/household/NGO — the municipality legally can't hold it. It surfaces as the two edges
disagreeing: Action→Funder strong, but City→Funder role = **referrer**.

In [ ]:
import re
blocked = [m for m in matches if "referrer route" in m["verdict"]]
print(f"{len(blocked)} of {len(matches)} Valdivia actions are blocked by the actor gap "
      f"({len(matches) - len(blocked)} are clean matches).\n")

# the verdict names the fund the action 'matches' but the city can't apply to:
def ref_funder(v):
    hit = re.search(r"\(([^)]+)\)", v)
    return hit.group(1) if hit else v
for funder, n in Counter(ref_funder(m["verdict"]) for m in blocked).most_common():
    print(f"  {n:>2} actions -> best fit is {funder}, but city is only a referrer")

print("\nExamples (the action, and the fund it 'matches' but can't hold):")
for m in blocked[:4]:
    print(f"  - {m['action'][:52]:<52} -> {ref_funder(m['verdict'])}")

print("\nInsight: without the role tag these 27 look like matches. The tag turns a false")
print("'yes' into the right move: refer your local firms, don't apply.")

### Insight 3 — Coordination tiers: why the transport *gap* is the best bundle

Every action gets a tier (`research/04`). The tier decides **what** you pool. Transport has *no
dedicated fund* (best fit 0.61) — but it's highly coordinated, which is exactly what makes it poolable.

In [ ]:
tier_order = ["highly_coordinated", "semi_coordinated", "idiosyncratic"]
mark = {"highly_coordinated":"[hi]", "semi_coordinated":"[semi]", "idiosyncratic":"[idio]"}

print(f"{'tier':<22} {'#actions':>8}   what you pool")
for t in tier_order:
    acts = [a for a in action_tiers if a["coordination_tier"] == t]
    print(f"{mark[t]:>6} {t:<15} {len(acts):>8}   {acts[0]['what_pooled']}")

trans = [a for a in action_tiers if a["sector"] == "Transportation"]
hi = sum(1 for a in trans if a["coordination_tier"] == "highly_coordinated")
print(f"\nInsight: transport has NO dedicated fund, yet {hi} of {len(trans)} transport actions are")
print("highly-coordinated. A supply gap for one city = a bundling opportunity for the region.")

### Insight 4 — Anchor economics: why pooling is the unlock (unit 11)

Co-finance capacity is wildly uneven inside a unit. The anchor carries the deal; the small,
transfer-dependent comunas ride its balance sheet (`research/05`).

In [ ]:
unit11 = sorted((u for u in units if u["unit_id"] == "11"),
                key=lambda r: float(r["cofinance_score"] or 0), reverse=True)

print(f"{'comuna':<14} {'co-finance':>10}  {'role':<8}")
for u in unit11:
    role = "ANCHOR" if u["is_anchor"] == "True" else "member"
    print(f"{u['comuna']:<14} {u['cofinance_score'] or 'n/a':>10}  {role:<8}")

scores = [float(u["cofinance_score"]) for u in unit11 if u["cofinance_score"]]
print(f"\nInsight: the anchor's co-finance ({max(scores):.0f}) is {max(scores)/min(scores):.0f}x the weakest "
      f"member ({min(scores):.0f}).")
print("Alone the small comunas can't clear a ticket. Pooled behind Valdivia, they can.")

### Insight 5 — Who can lead the country: anchors vs passengers

Nationally, which comunas can *anchor* a deal, and which can only ride along?

In [ ]:
lead      = [u for u in units if u["is_anchor"] == "True" and u["unit_viable_anchor"] == "True"]
passenger = [u for u in units if u["is_anchor"] != "True" and u["unit_viable_anchor"] == "True"]
stranded  = [u for u in units if u["unit_viable_anchor"] != "True"]
viable_units = {u["unit_id"] for u in units if u["unit_viable_anchor"] == "True"}
all_units    = {u["unit_id"] for u in units}

print(f"  {len(lead):>3} comunas can lead a pool (viable anchors)")
print(f"  {len(passenger):>3} 'passenger' comunas — can't lead alone, but a viable unit pulls them in")
print(f"  {len(stranded):>3} comunas in anchor-less units — served individually by grant/TA")
print(f"\n  {len(viable_units)} of {len(all_units)} units have a viable anchor")
print("\nInsight: only ~1 in 5 comunas can anchor. Pooling is what lets the other four reach deal-sized money.")

### Insight 6 — What we score vs. what we flag (the honesty)

The discipline a funder actually trusts: score what's in the data, flag what isn't.

In [ ]:
# Hole 1: amounts. Adequacy stays red because fund amounts aren't a column we can match on.
amount_cols = [c for c in funders[0] if "amount" in c.lower() or "clp" in c.lower()]
print(f"  funder amount columns available: {amount_cols or 'NONE'}  -> adequacy stays flagged (not faked)")

# Hole 2: coverage. Units cover 345 comunas, but only the ones with capacity data can be anchor-scored.
in_units = {u["locode"] for u in units}
scored   = {c["locode"] for c in capacity}
print(f"  comunas in units: {len(in_units)} · with capacity scores: {len(scored)} "
      f"· {len(in_units - scored)} unscored (no co-finance signal yet)")

print("\nInsight: the bundles are structurally poolable, but no dollar ticket is computed yet.")
print("'Green scored, red flagged' is the line that makes the output trustworthy.")